# Arrays de diferencias y sumas de prefijos 2D

Las sumas de prefijos y las diferencias son operaciones inversas. Con el array de diferencias $d_i = a_i - a_{i-1}$, sumar $v$ a un tramo $[i, j)$ solo toca dos posiciones ($d_i \mathrel{+}= v$, $d_j \mathrel{-}= v$), y los prefijos de $d$ reconstruyen el array: $q$ actualizaciones cuestan $O(n + q)$ en vez de $O(n \cdot q)$. En una matriz, el prefijo $P_{i,j}$ suma el rectángulo $[0, i) \times [0, j)$ y cualquier submatriz sale de cuatro valores por inclusión-exclusión:

$$S(f_1, c_1, f_2, c_2) = P_{f_2,c_2} - P_{f_1,c_2} - P_{f_2,c_1} + P_{f_1,c_1}$$

La diferencia 2D hace lo mismo al revés: sumar en un rectángulo toca cuatro esquinas. Este notebook amplía *Sumas de prefijos* con aplicaciones de ambas ideas.

## 1. Reservas de plazas en vuelos

Una aerolínea tiene los vuelos numerados de 1 a $n$ entre islas. Cada reserva $(primero, último, plazas)$ ocupa ese número de plazas en todos los vuelos del intervalo cerrado. Calcular las plazas reservadas por vuelo con un array de diferencias y comprobarlo contra aplicar cada reserva vuelo a vuelo.

In [1]:
import random
from itertools import accumulate

import numpy as np


def plazas_por_vuelo(n, reservas):
    d = [0] * (n + 2)
    for primero, ultimo, plazas in reservas:
        d[primero] += plazas
        d[ultimo + 1] -= plazas
    return list(accumulate(d[1:n + 1]))


def plazas_por_vuelo_bruto(n, reservas):
    total = [0] * n
    for primero, ultimo, plazas in reservas:
        for v in range(primero, ultimo + 1):
            total[v - 1] += plazas
    return total


reservas = [(1, 2, 10), (2, 3, 20), (2, 5, 25), (4, 4, 5)]
print(plazas_por_vuelo(5, reservas))

random.seed(2)
for _ in range(1000):
    n = random.randint(1, 15)
    reservas = []
    for _ in range(random.randint(0, 8)):
        a, b = sorted(random.choices(range(1, n + 1), k=2))
        reservas.append((a, b, random.randint(1, 30)))
    assert plazas_por_vuelo(n, reservas) == plazas_por_vuelo_bruto(n, reservas)

[10, 55, 45, 30, 25]


## 2. ¿Cabe todo el mundo en la guagua?

Una guagua de capacidad $C$ recorre paradas numeradas en orden. Cada viaje $(personas, sube, baja)$ ocupa asientos desde la parada `sube` hasta justo antes de `baja`. Decidir si en algún tramo se supera la capacidad y, si es así, devolver la primera parada donde ocurre. Comprobarlo contra una simulación parada a parada.

In [2]:
def primera_parada_llena(capacidad, viajes):
    if not viajes:
        return None
    n = max(baja for _, _, baja in viajes)
    d = [0] * (n + 1)
    for personas, sube, baja in viajes:
        d[sube] += personas
        d[baja] -= personas
    return next((parada for parada, ocupados in enumerate(accumulate(d)) if ocupados > capacidad), None)


def primera_parada_llena_bruto(capacidad, viajes):
    for parada in range(max((b for *_, b in viajes), default=0)):
        if sum(p for p, s, b in viajes if s <= parada < b) > capacidad:
            return parada
    return None


viajes = [(12, 0, 4), (20, 2, 6), (15, 5, 8), (10, 3, 5)]
for capacidad in (55, 50, 40):
    parada = primera_parada_llena(capacidad, viajes)
    print(f"Capacidad {capacidad}: {'cabe todo el mundo' if parada is None else f'se llena en la parada {parada}'}")

for _ in range(1000):
    viajes = []
    for _ in range(random.randint(0, 6)):
        s = random.randint(0, 8)
        viajes.append((random.randint(1, 10), s, random.randint(s + 1, 10)))
    c = random.randint(5, 25)
    assert primera_parada_llena(c, viajes) == primera_parada_llena_bruto(c, viajes)

Capacidad 55: cabe todo el mundo
Capacidad 50: cabe todo el mundo
Capacidad 40: se llena en la parada 3


## 3. Sumar progresiones en un tramo

Una actualización suma $1, 2, 3, \dots$ a las posiciones $i, i+1, \dots, j-1$ (por ejemplo, un descuento que crece cada día de una oferta). Resolverla con diferencias de segundo orden: la segunda diferencia de una progresión aritmética es constante, así que basta tocar tres posiciones y acumular dos veces. Comprobarlo contra la fuerza bruta.

In [3]:
def suma_progresiones(n, tramos):
    d2 = [0] * (n + 2)
    for i, j in tramos:
        longitud = j - i
        d2[i] += 1
        d2[j] -= longitud + 1
        d2[j + 1] += longitud
    return list(accumulate(accumulate(d2)))[:n]


def suma_progresiones_bruto(n, tramos):
    a = [0] * n
    for i, j in tramos:
        for k in range(i, j):
            a[k] += k - i + 1
    return a


print(suma_progresiones(8, [(1, 5), (3, 8)]))

for _ in range(1000):
    n = random.randint(1, 12)
    tramos = [tuple(sorted(random.sample(range(n + 1), 2))) for _ in range(random.randint(0, 6))]
    assert suma_progresiones(n, tramos) == suma_progresiones_bruto(n, tramos)

[0, 1, 2, 4, 6, 3, 4, 5]


## 4. Mínimo de operaciones para construir un array

Partiendo de ceros, cada operación suma 1 a un tramo contiguo. Calcular el mínimo número de operaciones para obtener un array objetivo de números no negativos: cada operación sube una posición del array de diferencias y baja otra, así que la respuesta es la suma de las diferencias positivas $\sum \max(0, a_i - a_{i-1})$. Comprobarlo contra una búsqueda en anchura sobre estados en arrays pequeños.

In [4]:
from collections import deque


def operaciones_minimas(objetivo):
    return sum(max(0, b - a) for a, b in zip([0, *objetivo], objetivo))


def operaciones_minimas_bfs(objetivo):
    n = len(objetivo)
    inicio, meta = (0,) * n, tuple(objetivo)
    distancia = {inicio: 0}
    cola = deque([inicio])
    while cola:
        estado = cola.popleft()
        if estado == meta:
            return distancia[estado]
        for i in range(n):
            for j in range(i + 1, n + 1):
                nuevo = tuple(x + (i <= k < j) for k, x in enumerate(estado))
                if all(x <= m for x, m in zip(nuevo, meta)) and nuevo not in distancia:
                    distancia[nuevo] = distancia[estado] + 1
                    cola.append(nuevo)


objetivo = [1, 2, 3, 2, 1, 3]
print(f"{objetivo}: {operaciones_minimas(objetivo)} operaciones")

for _ in range(200):
    objetivo = [random.randint(0, 3) for _ in range(random.randint(1, 5))]
    assert operaciones_minimas(objetivo) == operaciones_minimas_bfs(objetivo)

[1, 2, 3, 2, 1, 3]: 5 operaciones


## 5. Consultas sobre un mapa de población

Una cuadrícula de $12 \times 16$ celdas guarda los habitantes de cada celda de una isla. Crear una clase `MapaSumas` que precalcule los prefijos 2D con NumPy (`cumsum` en los dos ejes y una fila y columna de ceros delante) y responda en $O(1)$ cuántos habitantes hay en cualquier rectángulo. Comprobarlo con `sum` sobre el corte.

In [5]:
class MapaSumas:
    def __init__(self, matriz):
        m = np.asarray(matriz)
        self.p = np.zeros((m.shape[0] + 1, m.shape[1] + 1), dtype=m.dtype)
        self.p[1:, 1:] = m.cumsum(axis=0).cumsum(axis=1)

    def suma(self, f1, c1, f2, c2):
        """Suma de las filas [f1, f2) y columnas [c1, c2)."""
        p = self.p
        return int(p[f2, c2] - p[f1, c2] - p[f2, c1] + p[f1, c1])


rng = np.random.default_rng(5)
poblacion = rng.poisson(30, size=(12, 16))
poblacion[4:8, 5:10] += rng.poisson(400, size=(4, 5))  # un núcleo urbano
mapa = MapaSumas(poblacion)
print("Total:", mapa.suma(0, 0, 12, 16))
print("Núcleo urbano (filas 4-7, columnas 5-9):", mapa.suma(4, 5, 8, 10))
print("Mitad norte:", mapa.suma(0, 0, 6, 16), " mitad sur:", mapa.suma(6, 0, 12, 16))

for _ in range(2000):
    f1, f2 = sorted(rng.integers(0, 13, size=2))
    c1, c2 = sorted(rng.integers(0, 17, size=2))
    assert mapa.suma(f1, c1, f2, c2) == poblacion[f1:f2, c1:c2].sum()

Total: 13881
Núcleo urbano (filas 4-7, columnas 5-9): 8689
Mitad norte: 6990  mitad sur: 6891


## 6. Diferencias 2D: cobertura de aspersores

Cada aspersor de una finca riega un rectángulo de celdas. Para sumar 1 en un rectángulo $[f_1, f_2) \times [c_1, c_2)$ basta tocar cuatro esquinas del array de diferencias: $+1$ en $(f_1, c_1)$ y $(f_2, c_2)$, $-1$ en $(f_1, c_2)$ y $(f_2, c_1)$. Los prefijos 2D reconstruyen cuántos aspersores riegan cada celda. Contar las celdas secas y las regadas dos o más veces, y comprobarlo contra sumar rectángulo a rectángulo.

In [6]:
def cobertura(filas, columnas, rectangulos):
    d = np.zeros((filas + 1, columnas + 1), dtype=int)
    for f1, c1, f2, c2 in rectangulos:
        d[f1, c1] += 1
        d[f1, c2] -= 1
        d[f2, c1] -= 1
        d[f2, c2] += 1
    return d.cumsum(axis=0).cumsum(axis=1)[:filas, :columnas]


def cobertura_bruta(filas, columnas, rectangulos):
    m = np.zeros((filas, columnas), dtype=int)
    for f1, c1, f2, c2 in rectangulos:
        m[f1:f2, c1:c2] += 1
    return m


aspersores = [(0, 0, 4, 5), (2, 3, 7, 9), (5, 0, 8, 4), (1, 7, 3, 12), (6, 8, 8, 12)]
riego = cobertura(8, 12, aspersores)
for fila in riego:
    print(" ".join(".123456789"[x] for x in fila))
print("Celdas secas:", int((riego == 0).sum()), "  regadas dos o más veces:", int((riego >= 2).sum()))

for _ in range(500):
    filas, columnas = rng.integers(1, 10, size=2)
    rects = []
    for _ in range(rng.integers(0, 8)):
        f1, f2 = sorted(rng.integers(0, filas + 1, size=2))
        c1, c2 = sorted(rng.integers(0, columnas + 1, size=2))
        rects.append((f1, c1, f2, c2))
    assert np.array_equal(cobertura(filas, columnas, rects), cobertura_bruta(filas, columnas, rects))

1 1 1 1 1 . . . . . . .
1 1 1 1 1 . . 1 1 1 1 1
1 1 1 2 2 1 1 2 2 1 1 1
1 1 1 2 2 1 1 1 1 . . .
. . . 1 1 1 1 1 1 . . .
1 1 1 2 1 1 1 1 1 . . .
1 1 1 2 1 1 1 1 2 1 1 1
1 1 1 1 . . . . 1 1 1 1
Celdas secas: 25   regadas dos o más veces: 9


## 7. Máximo de las submatrices de lado k

En un mapa de radiación solar (kWh/m² por celda), elegir dónde poner una planta fotovoltaica cuadrada de lado $k$ para maximizar la radiación total. Calcular las sumas de todas las ventanas $k \times k$ a la vez restando cortes de la matriz de prefijos con NumPy, en $O(filas \cdot columnas)$, y comprobarlo contra sumar cada ventana. Repetirlo para varios $k$.

In [7]:
def sumas_ventanas(m, k):
    p = np.zeros((m.shape[0] + 1, m.shape[1] + 1), dtype=m.dtype)
    p[1:, 1:] = m.cumsum(axis=0).cumsum(axis=1)
    return p[k:, k:] - p[:-k, k:] - p[k:, :-k] + p[:-k, :-k]


def sumas_ventanas_bruto(m, k):
    filas, columnas = m.shape
    return np.array([[m[i:i + k, j:j + k].sum() for j in range(columnas - k + 1)] for i in range(filas - k + 1)])


radiacion = rng.integers(40, 80, size=(10, 14))
radiacion[2:6, 8:12] += 25  # ladera orientada al sur
print(f"{'k':>2} {'máximo':>8} {'posición':>10}")
for k in (1, 2, 3, 4, 5):
    s = sumas_ventanas(radiacion, k)
    f, c = np.unravel_index(s.argmax(), s.shape)
    print(f"{k:>2} {int(s.max()):>8} {f'({f}, {c})':>10}")
    assert np.array_equal(s, sumas_ventanas_bruto(radiacion, k))

 k   máximo   posición
 1      103    (3, 10)
 2      373     (2, 9)
 3      769     (2, 9)
 4     1316     (2, 8)
 5     1915     (1, 8)


## 8. Mayor cuadrado con suma acotada

Encontrar el lado del mayor cuadrado cuya suma no pasa de un umbral (por ejemplo, la mayor parcela cuadrada cuyo precio total cabe en un presupuesto). Si todos los valores son no negativos, la propiedad "existe un cuadrado de lado $k$ que cabe" es monótona en $k$, así que se puede hacer búsqueda binaria usando las sumas de ventanas. Comprobarlo contra probar todos los lados.

In [8]:
def mayor_lado(m, umbral):
    def cabe(k):
        return k == 0 or (sumas_ventanas(m, k) <= umbral).any()

    bajo, alto = 0, min(m.shape)
    while bajo < alto:
        medio = (bajo + alto + 1) // 2
        if cabe(medio):
            bajo = medio
        else:
            alto = medio - 1
    return bajo


def mayor_lado_bruto(m, umbral):
    return max([k for k in range(1, min(m.shape) + 1) if (sumas_ventanas_bruto(m, k) <= umbral).any()], default=0)


precios = rng.integers(1, 10, size=(9, 11))
for presupuesto in (5, 40, 100, 250):
    print(f"Presupuesto {presupuesto:>3}: lado {mayor_lado(precios, presupuesto)}")

for _ in range(300):
    m = rng.integers(0, 10, size=rng.integers(1, 7, size=2))
    umbral = int(rng.integers(0, 60))
    assert mayor_lado(m, umbral) == mayor_lado_bruto(m, umbral)

Presupuesto   5: lado 1
Presupuesto  40: lado 3
Presupuesto 100: lado 4
Presupuesto 250: lado 7


## 9. Submatrices que suman un objetivo

Contar las submatrices (rectángulos contiguos) cuya suma es exactamente $t$, con valores negativos incluidos. Fijar cada par de filas $(f_1, f_2)$, comprimir las columnas en un array de sumas con los prefijos verticales y contar en él los tramos que suman $t$ con un diccionario de prefijos. Cuesta $O(filas^2 \cdot columnas)$ frente a $O(filas^3 \cdot columnas^3)$. Comprobarlo contra la fuerza bruta.

In [9]:
from collections import Counter


def tramos_con_suma(a, t):
    vistos, total, cuenta = Counter([0]), 0, 0
    for x in a:
        total += x
        cuenta += vistos[total - t]
        vistos[total] += 1
    return cuenta


def submatrices_con_suma(m, t):
    filas = len(m)
    vertical = np.vstack([np.zeros(m.shape[1], dtype=int), m.cumsum(axis=0)])
    return sum(tramos_con_suma((vertical[f2] - vertical[f1]).tolist(), t) for f1 in range(filas) for f2 in range(f1 + 1, filas + 1))


def submatrices_con_suma_bruto(m, t):
    filas, columnas = m.shape
    return sum(
        m[f1:f2, c1:c2].sum() == t
        for f1 in range(filas) for f2 in range(f1 + 1, filas + 1)
        for c1 in range(columnas) for c2 in range(c1 + 1, columnas + 1)
    )


m = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])
print("Submatrices que suman 0:", submatrices_con_suma(m, 0), "  que suman 3:", submatrices_con_suma(m, 3))

for _ in range(300):
    m = rng.integers(-2, 3, size=rng.integers(1, 6, size=2))
    t = int(rng.integers(-2, 3))
    assert submatrices_con_suma(m, t) == submatrices_con_suma_bruto(m, t)

Submatrices que suman 0: 4   que suman 3: 6


## 10. Submatriz de suma máxima

Encontrar el rectángulo de suma máxima en una matriz con valores positivos y negativos (Kadane 2D): para cada par de filas, comprimir las columnas como en el ejercicio anterior y aplicar Kadane al array resultante. Devolver la suma y las coordenadas, y comprobarlo contra la fuerza bruta con prefijos 2D.

In [10]:
def kadane(a):
    mejor, inicio_mejor, fin_mejor = a[0], 0, 1
    actual, inicio = 0, 0
    for j, x in enumerate(a):
        if actual <= 0:
            actual, inicio = x, j
        else:
            actual += x
        if actual > mejor:
            mejor, inicio_mejor, fin_mejor = actual, inicio, j + 1
    return mejor, inicio_mejor, fin_mejor


def submatriz_maxima(m):
    filas = m.shape[0]
    vertical = np.vstack([np.zeros(m.shape[1], dtype=int), m.cumsum(axis=0)])
    mejor = None
    for f1 in range(filas):
        for f2 in range(f1 + 1, filas + 1):
            suma, c1, c2 = kadane((vertical[f2] - vertical[f1]).tolist())
            if mejor is None or suma > mejor[0]:
                mejor = (suma, f1, c1, f2, c2)
    return mejor


def submatriz_maxima_bruta(m):
    mapa = MapaSumas(m)
    filas, columnas = m.shape
    return max(
        mapa.suma(f1, c1, f2, c2)
        for f1 in range(filas) for f2 in range(f1 + 1, filas + 1)
        for c1 in range(columnas) for c2 in range(c1 + 1, columnas + 1)
    )


beneficios = np.array([
    [1, -2, -1, 4],
    [-8, 3, 4, 2],
    [3, 8, 10, -8],
    [-4, -1, 1, 7],
])
suma, f1, c1, f2, c2 = submatriz_maxima(beneficios)
print(f"Suma {suma} en filas {f1}..{f2 - 1}, columnas {c1}..{c2 - 1}:")
print(beneficios[f1:f2, c1:c2])

for _ in range(300):
    m = rng.integers(-9, 10, size=rng.integers(1, 6, size=2))
    suma, f1, c1, f2, c2 = submatriz_maxima(m)
    assert suma == submatriz_maxima_bruta(m) == m[f1:f2, c1:c2].sum()

Suma 27 en filas 0..3, columnas 1..3:
[[-2 -1  4]
 [ 3  4  2]
 [ 8 10 -8]
 [-1  1  7]]


## 11. Desenfoque de caja con la imagen integral

En visión por computador la matriz de prefijos se llama imagen integral. Suavizar una imagen en escala de grises sustituyendo cada píxel por la media de su vecindario $(2r+1) \times (2r+1)$, recortado en los bordes, con un coste por píxel que no depende de $r$. Comprobarlo contra calcular cada media con un corte y mostrar el resultado en texto.

In [11]:
def desenfoque(imagen, r):
    filas, columnas = imagen.shape
    p = np.zeros((filas + 1, columnas + 1))
    p[1:, 1:] = imagen.cumsum(axis=0).cumsum(axis=1)
    f1 = np.clip(np.arange(filas) - r, 0, filas)[:, None]
    f2 = np.clip(np.arange(filas) + r + 1, 0, filas)[:, None]
    c1 = np.clip(np.arange(columnas) - r, 0, columnas)[None, :]
    c2 = np.clip(np.arange(columnas) + r + 1, 0, columnas)[None, :]
    suma = p[f2, c2] - p[f1, c2] - p[f2, c1] + p[f1, c1]
    return suma / ((f2 - f1) * (c2 - c1))


def desenfoque_bruto(imagen, r):
    filas, columnas = imagen.shape
    return np.array([[imagen[max(0, i - r):i + r + 1, max(0, j - r):j + r + 1].mean() for j in range(columnas)] for i in range(filas)])


imagen = np.zeros((9, 18))
imagen[2:7, 3:8] = 9
imagen[4, 11:16] = 9
suave = desenfoque(imagen, 1)
for original, borrosa in zip(imagen, suave):
    print("".join(" .:-=+*#%@"[int(x)] for x in original), "  ", "".join(" .:-=+*#%@"[round(x)] for x in borrosa))

for r in range(4):
    for _ in range(20):
        img = rng.integers(0, 256, size=rng.integers(1, 12, size=2)).astype(float)
        assert np.allclose(desenfoque(img, r), desenfoque_bruto(img, r))

                                        
                        .:---:.         
   @@@@@                :=***=:         
   @@@@@                -*@@@*- .:---:. 
   @@@@@   @@@@@        -*@@@*- .:---:. 
   @@@@@                -*@@@*- .:---:. 
   @@@@@                :=***=:         
                        .:---:.         
                                        
